# ReST — spectral model selection for VLMs

In [1]:
import os, csv, logging, warnings, numpy as np, pandas as pd, torch, open_clip
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from scipy.stats import weightedtau
from tqdm.auto import tqdm

# ---- quiet -------------------------------------------------------------
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
for _n in ("open_clip", "PIL", "torch", "urllib3", "huggingface_hub", "datasets"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers import logging as _hf; _hf.set_verbosity_error()
except Exception:
    pass
try:
    torch.set_warn_always(False)
except Exception:
    pass

try:
    from transformers.tokenization_utils_base import PreTrainedTokenizerBase as _P
    if not hasattr(_P, "batch_encode_plus"):
        _P.batch_encode_plus = lambda self, b, **kw: self(b, **kw)
except Exception:
    pass

# ---- config 
ALPHA        = 0.67   
NUM_SAMPLES  = 100    # unlabelled target images used for scoring
SEED         = 42
BATCH_SIZE   = 32
NUM_WORKERS  = 2      
GT_DIR       = "."
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

GT_CSV_BY_DATASET = {"cifar100": "cifar100_gt.csv"}

## 1. Stable rank

In [2]:
def stable_rank(X: torch.Tensor) -> tuple:
    """
    SR(A) = ||Sigma||_F^2 / ||Sigma||_2^2 = sum_i sigma_i^2 / sigma_1^2.
    Also returns the top-k normalised singular values for spectrum plots.
    """
    if X is None:
        return float("nan"), []
    if X.dim() == 1:
        return 1.0, [1.0]
    if X.dim() > 2:
        X = X.view(X.size(0), -1)

    S = torch.linalg.svdvals(X.to(torch.float32))
    sr = float((S.pow(2).sum() / (S[0] ** 2 + 1e-12)).item())
    return sr, (S / (S.sum() + 1e-12))[:100].cpu().tolist()


def attach_preprocess(ds, preprocess):
    """Attach the CLIP preprocess to the dataset or to the wrapped Subset."""
    if hasattr(ds, "transform"):
        ds.transform = preprocess
    elif hasattr(ds, "dataset") and hasattr(ds.dataset, "transform"):
        ds.dataset.transform = preprocess

## 2. Datasets and model hub

In [3]:
def _sample_subset(ds, k, seed):
    """K random target images. Labels are never read."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    idx = np.random.choice(len(ds), size=min(k, len(ds)), replace=False).tolist()
    return torch.utils.data.Subset(ds, idx)


def setup_cifar100(num_samples=NUM_SAMPLES, seed=SEED):
    ds = datasets.CIFAR100(root="./data", train=False, download=True,
                           transform=transforms.ToTensor())
    subset = _sample_subset(ds, num_samples, seed)
    print(f"CIFAR-100: Sampled {len(subset)} images (seed={seed})")
    return subset, [c.replace("_", " ") for c in ds.classes]


DATASET_SETUPS = {"cifar100": setup_cifar100}


def load_model_hub(dataset_key, gt_dir=GT_DIR):
    """
    Hub + ground-truth accuracy from <dataset>_gt.csv. Required columns:
    vlm_model, vlm_pretraining_dataset, imagenet_top1_accuracy, top1_accuracy
    """
    path = os.path.join(gt_dir, GT_CSV_BY_DATASET[dataset_key.lower()])
    if not os.path.exists(path):
        raise FileNotFoundError(f"Ground-truth CSV not found: {path}")
    df = pd.read_csv(path)
    need = ["vlm_model", "vlm_pretraining_dataset", "imagenet_top1_accuracy", "top1_accuracy"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing column(s): {missing}")
    out = [(r.vlm_model, r.vlm_pretraining_dataset,
            r.imagenet_top1_accuracy, r.top1_accuracy) for r in df.itertuples()]
    print(f"Loaded {len(out)} models + ground truth from {path}")
    return out

## 3. ReST score

In [4]:
@torch.no_grad()
def compute_rest(model, dataset, class_names, model_name, dl_num_workers=NUM_WORKERS):
    """Stable rank of the vision and language projection activations."""
    device = next(model.parameters()).device
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=dl_num_workers, pin_memory=True)

    tokenizer = open_clip.get_tokenizer(model_name)
    text_tokens = tokenizer([f"a photo of {n}" for n in class_names]).to(device)

    feats = []
    for images, _ in tqdm(dataloader, desc="    vision projection", leave=False):
        feats.append(model.encode_image(images.to(device, non_blocking=True)).detach().cpu())
    vision_acts = torch.cat(feats, dim=0) if feats else None
    lang_acts = model.encode_text(text_tokens).detach().cpu()

    sv, sv_spec = stable_rank(vision_acts)
    sl, sl_spec = stable_rank(lang_acts)

    return {
        "vision_projection_activation_stable_rank": sv,
        "language_projection_activation_stable_rank": sl,
        "rest_score": ALPHA * sv + (1 - ALPHA) * sl,
        "vision_projection_activation_svd_top100": sv_spec,
        "language_projection_activation_svd_top100": sl_spec,
    }

## 4. Run

In [ ]:
os.makedirs("rest_metrics", exist_ok=True)
all_rows = []

for key, setup in DATASET_SETUPS.items():
    dataset, class_names = setup()
    hub = load_model_hub(key)
    print(f"\n{'='*70}\nReST on {key}  ({NUM_SAMPLES} unlabelled samples)\n{'='*70}")

    results = []
    for i, (name, pretrained, inb, gt) in enumerate(hub, 1):
        model = None
        try:
            print(f"  [{i:2d}/{len(hub)}] {name}/{pretrained}", flush=True)
            model, _, preprocess = open_clip.create_model_and_transforms(
                name, pretrained=pretrained, device=DEVICE)
            model.eval()
            attach_preprocess(dataset, preprocess)

            m = compute_rest(model, dataset, class_names, name)
            results.append(dict(dataset=key, vlm_model=name, vlm_pretraining_dataset=pretrained,
                                imagenet_top1_accuracy=inb, groundtruth_top1_accuracy=gt, **m))
            print(f"        ReST={m['rest_score']:.3f} "
                  f"(v={m['vision_projection_activation_stable_rank']:.3f}, "
                  f"l={m['language_projection_activation_stable_rank']:.3f})  gt={gt:.2f}%", flush=True)
        except Exception as e:
            print(f"        x {type(e).__name__}: {e}")
        finally:
            del model
            torch.cuda.empty_cache()

    if results:
        pd.DataFrame(results).to_csv(f"rest_metrics/{key}_rest_{NUM_SAMPLES}samples.csv", index=False)
        print(f"  -> rest_metrics/{key}_rest_{NUM_SAMPLES}samples.csv ({len(results)} models)")
        all_rows += results

df = pd.DataFrame(all_rows)
df.to_csv("rest_metrics/rest_all.csv", index=False)
df[["dataset", "vlm_model", "vlm_pretraining_dataset",
    "vision_projection_activation_stable_rank",
    "language_projection_activation_stable_rank",
    "rest_score", "groundtruth_top1_accuracy"]].head()

CIFAR-100: Sampled 100 images (seed=42)
Loaded 20 models + ground truth from ./cifar100_gt.csv

ReST on cifar100  (100 unlabelled samples)
  [ 1/20] ViT-L-14-quickgelu/dfn2b


    vision projection:   0%|          | 0/4 [00:00<?, ?it/s]

        ReST=1.711 (v=1.857, l=1.415)  gt=87.03%
  [ 2/20] ViT-H-14-quickgelu/metaclip_fullcc


    vision projection:   0%|          | 0/4 [00:00<?, ?it/s]

        ReST=1.327 (v=1.353, l=1.274)  gt=79.80%
  [ 3/20] ViT-bigG-14/laion2b_s39b_b160k


...

        ReST=1.215 (v=1.192, l=1.261)  gt=68.25%
  [16/20] ViT-B-16/datacomp_xl_s13b_b90k


    vision projection:   0%|          | 0/4 [00:00<?, ?it/s]

        ReST=1.853 (v=2.085, l=1.383)  gt=79.72%
  [17/20] ViT-L-14/openai


    vision projection:   0%|          | 0/4 [00:00<?, ?it/s]

        ReST=1.244 (v=1.231, l=1.269)  gt=70.23%
  [18/20] RN50x64/openai


## 5. Vision vs language stable rank

In [ ]:
g = df[df.dataset == "cifar100"]

fig, ax = plt.subplots(figsize=(5.5, 4.6))
x = g.vision_projection_activation_stable_rank
y = g.language_projection_activation_stable_rank
s = ax.scatter(x, y, c=g.groundtruth_top1_accuracy, cmap="viridis",
               s=110, edgecolor="k", linewidth=.5)
xs = np.linspace(x.min(), x.max(), 50)
for q in np.quantile(g.rest_score, [.25, .5, .75]):
    ax.plot(xs, (q - ALPHA * xs) / (1 - ALPHA), "--", c="grey", lw=.8, alpha=.6)
ax.set(xlabel="vision projection stable rank",
       ylabel="language projection stable rank", title=f"cifar100  (n={len(g)})")
ax.set_ylim(y.min() - .05, y.max() + .05)
plt.colorbar(s, ax=ax, label="ground-truth top-1 (%)")
fig.suptitle(f"ReST plane, alpha={ALPHA} (dashed = iso-score lines)", y=1.02)
plt.tight_layout(); plt.savefig("rest_plane.png", dpi=150, bbox_inches="tight"); plt.show()

## 6. Correlation with ground truth

In [ ]:
def wtau(x, y):
    return weightedtau(np.ascontiguousarray(x, float).copy(),
                       np.ascontiguousarray(y, float).copy())[0]

out = []
for key, g in df.groupby("dataset"):
    gt = g.groundtruth_top1_accuracy.values
    out.append({
        "dataset": key,
        "ReST": wtau(g.rest_score.values, gt),
        "selected": gt[int(np.argmax(g.rest_score.values))],
        "oracle": gt.max(), "random": gt.mean()})

output = pd.DataFrame(out)
output